In [1]:
# # 03 - Uji Kesepakatan Antarpenilai (Cohen's Kappa)
# Mengukur kesepakatan penilai untuk **aspek** (4 kelompok kata kunci) dan **sentimen** (1.000 data).
# 
# - Cohen's kappa untuk setiap pasangan penilai (1-2, 1-3, 2-3), keseluruhan dan per aspek, dengan selang kepercayaan bootstrap 95%
# - Fleiss' kappa untuk ketiga penilai sekaligus
# - Tafsiran skala Landis & Koch (1977)
# 
# Input : keluaran notebook 01 dan 02. Output: `hasil_validasi/hasil_kappa.xlsx`


# ===== SEL 1 (salin ke satu sel notebook) =====
import os, re, itertools, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

FOLDER_DATA  = "validasi data"             # berkas validasi_*.xlsx dan aspek_kalimat.csv
FOLDER_HASIL = "hasil_validasi"   # semua keluaran disimpan di sini
os.makedirs(FOLDER_HASIL, exist_ok=True)

SEED = 42
LABEL = ["positif", "negatif", "netral"]
ASPEK = ["wifi", "harga", "suasana", "pelayanan"]
N_PENILAI = 3

def tafsir_kappa(k):
    """Skala Landis & Koch (1977)."""
    if k is None or np.isnan(k): return "tidak dapat dihitung"
    if k < 0:     return "buruk (di bawah kebetulan)"
    if k <= 0.20: return "sangat rendah"
    if k <= 0.40: return "rendah"
    if k <= 0.60: return "sedang"
    if k <= 0.80: return "baik (substantial)"
    return "sangat baik (almost perfect)"

from sklearn.metrics import cohen_kappa_score, confusion_matrix

def fleiss_kappa(tabel):
    """tabel: (n_butir x n_kategori), tiap baris = jumlah penilai pada kategori; tiap baris berjumlah sama."""
    tabel = np.asarray(tabel, dtype=float)
    n = tabel.sum(axis=1)[0]
    P_i = ((tabel ** 2).sum(axis=1) - n) / (n * (n - 1))
    p_j = tabel.sum(axis=0) / tabel.sum()
    P_e = (p_j ** 2).sum()
    return np.nan if P_e == 1 else (P_i.mean() - P_e) / (1 - P_e)

def kappa_bootstrap(a, b, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    hasil = []
    for _ in range(n_boot):
        i = rng.integers(0, len(a), len(a))
        if len(set(a[i])) < 2 and len(set(b[i])) < 2: continue
        hasil.append(cohen_kappa_score(a[i], b[i]))
    return (np.percentile(hasil, 2.5), np.percentile(hasil, 97.5)) if hasil else (np.nan, np.nan)


# ## A. Kappa aspek


# ===== SEL 2 (salin ke satu sel notebook) =====
asp = pd.read_csv(f"{FOLDER_HASIL}/validasi_aspek_gabungan.csv")
kolom_a = [f"penilai_{p}" for p in range(1, N_PENILAI + 1)]
baris_a = []
for p, q in itertools.combinations(range(1, N_PENILAI + 1), 2):
    k = cohen_kappa_score(asp[f"penilai_{p}"], asp[f"penilai_{q}"])
    baris_a.append({"ukuran": f"Cohen penilai {p} vs {q}", "n_butir": len(asp), "kesepakatan": (asp[f"penilai_{p}"] == asp[f"penilai_{q}"]).mean(), "kappa": k, "tafsiran": tafsir_kappa(k)})
tab = np.array([[int((asp.loc[i, kolom_a] == c).sum()) for c in ASPEK] for i in asp.index])
kf = fleiss_kappa(tab)
baris_a.append({"ukuran": "Fleiss (3 penilai)", "n_butir": len(asp), "kesepakatan": np.nan, "kappa": kf, "tafsiran": tafsir_kappa(kf)})
kappa_aspek = pd.DataFrame(baris_a)
kappa_aspek


# > Catatan: bila semua penilai memilih label yang sama untuk semua butir (seperti pada 4 kelompok kata kunci ini), kesepakatan = 100% dan kappa = 1. Hanya 4 butir, jadi hasil ini bersifat indikatif.


# ## B. Kappa sentimen (1.000 data)


# ===== SEL 3 (salin ke satu sel notebook) =====
sen = pd.read_csv(f"{FOLDER_HASIL}/label_penilai_1000.csv")
kolom_s = [f"penilai_{p}" for p in range(1, N_PENILAI + 1)]
baris_k, baris_pa, matriks = [], [], {}
for p, q in itertools.combinations(range(1, N_PENILAI + 1), 2):
    sub = sen[[f"penilai_{p}", f"penilai_{q}", "aspek"]].dropna()
    a, b = sub[f"penilai_{p}"], sub[f"penilai_{q}"]
    k = cohen_kappa_score(a, b, labels=LABEL)
    lo, hi = kappa_bootstrap(a.values, b.values)
    baris_k.append({"pasangan": f"{p} vs {q}", "n": len(sub), "kesepakatan": (a == b).mean(), "kappa": k,
                    "ci95_bawah": lo, "ci95_atas": hi, "tafsiran": tafsir_kappa(k)})
    for asp_ in ASPEK:
        s2 = sub[sub["aspek"] == asp_]
        ka = cohen_kappa_score(s2.iloc[:, 0], s2.iloc[:, 1], labels=LABEL)
        baris_pa.append({"pasangan": f"{p} vs {q}", "aspek": asp_, "n": len(s2),
                         "kesepakatan": (s2.iloc[:, 0] == s2.iloc[:, 1]).mean(), "kappa": ka, "tafsiran": tafsir_kappa(ka)})
    matriks[f"{p}vs{q}"] = pd.DataFrame(confusion_matrix(a, b, labels=LABEL),
                                        index=[f"p{p}_{l}" for l in LABEL], columns=[f"p{q}_{l}" for l in LABEL])
kappa_sentimen = pd.DataFrame(baris_k)
kappa_per_aspek = pd.DataFrame(baris_pa)
kappa_sentimen


# ===== SEL 4 (salin ke satu sel notebook) =====
kappa_per_aspek.pivot(index="aspek", columns="pasangan", values="kappa").reindex(ASPEK).round(4)


# ===== SEL 5 (salin ke satu sel notebook) =====
for nama, m in matriks.items():
    print("Matriks konfusi penilai", nama); display(m); print()


# ===== SEL 6 (salin ke satu sel notebook) =====
# Fleiss' kappa: hanya butir yang dinilai ketiga penilai
lengkap = sen.dropna(subset=kolom_s)
tab = np.array([[int((lengkap.loc[i, kolom_s] == l).sum()) for l in LABEL] for i in lengkap.index])
kf_s = fleiss_kappa(tab)
print(f"Fleiss' kappa sentimen (n={len(lengkap)} butir lengkap, 3 penilai): {kf_s:.4f} -> {tafsir_kappa(kf_s)}")

# Fleiss per aspek
baris_f = []
for asp_ in ASPEK:
    sub = lengkap[lengkap["aspek"] == asp_]
    t = np.array([[int((sub.loc[i, kolom_s] == l).sum()) for l in LABEL] for i in sub.index])
    baris_f.append({"aspek": asp_, "n": len(sub), "fleiss_kappa": fleiss_kappa(t), "tafsiran": tafsir_kappa(fleiss_kappa(t))})
fleiss_aspek = pd.DataFrame(baris_f)
fleiss_aspek


# ## Simpan


# ===== SEL 7 (salin ke satu sel notebook) =====
with pd.ExcelWriter(f"{FOLDER_HASIL}/hasil_kappa.xlsx") as xw:
    kappa_aspek.to_excel(xw, sheet_name="kappa_aspek", index=False)
    kappa_sentimen.to_excel(xw, sheet_name="kappa_sentimen", index=False)
    kappa_per_aspek.to_excel(xw, sheet_name="kappa_sentimen_per_aspek", index=False)
    pd.DataFrame([{"n": len(lengkap), "fleiss_kappa": kf_s, "tafsiran": tafsir_kappa(kf_s)}]).to_excel(xw, sheet_name="fleiss_sentimen", index=False)
    fleiss_aspek.to_excel(xw, sheet_name="fleiss_per_aspek", index=False)
    for nama, m in matriks.items():
        m.to_excel(xw, sheet_name="konfusi_" + nama)
print("Tersimpan: hasil_kappa.xlsx")

Matriks konfusi penilai 1vs2


,p2_positif,p2_negatif,p2_netral
p1_positif,562,3,38
p1_negatif,12,270,14
p1_netral,27,16,58



Matriks konfusi penilai 1vs3


,p3_positif,p3_negatif,p3_netral
p1_positif,590,3,10
p1_negatif,33,258,5
p1_netral,70,25,6



Matriks konfusi penilai 2vs3


,p3_positif,p3_negatif,p3_netral
p2_positif,580,16,5
p2_negatif,36,249,4
p2_netral,77,21,12



Fleiss' kappa sentimen (n=1000 butir lengkap, 3 penilai): 0.7287 -> baik (substantial)
Tersimpan: hasil_kappa.xlsx
